In [0]:
from pathlib import Path

import pandas as pd
from pyspark.sql.functions import current_timestamp, lit


def find_repository_root(starting_path: Path) -> Path:
    """Locate the repository root without hardcoding a user workspace path."""

    possible_paths = [starting_path, *starting_path.parents]

    for candidate in possible_paths:
        sales_file = (
            candidate
            / "sales_pipeline"
            / "data"
            / "raw"
            / "sales_raw.csv"
        )

        if sales_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate sales_pipeline/data/raw/sales_raw.csv "
        f"from starting path {starting_path}"
    )


repository_root = find_repository_root(Path.cwd())

sales_csv_path = (
    repository_root
    / "sales_pipeline"
    / "data"
    / "raw"
    / "sales_raw.csv"
)

products_csv_path = (
    repository_root
    / "sales_pipeline"
    / "data"
    / "raw"
    / "products_raw.csv"
)

print(f"Repository root: {repository_root}")
print(f"Sales source: {sales_csv_path}")
print(f"Products source: {products_csv_path}")

In [0]:
sales_pandas_df = pd.read_csv(sales_csv_path)
products_pandas_df = pd.read_csv(products_csv_path)

print(f"Sales source rows: {len(sales_pandas_df):,}")
print(f"Products source rows: {len(products_pandas_df):,}")

In [0]:
sales_bronze_df = (
    spark.createDataFrame(sales_pandas_df)
    .withColumn("source_system", lit("repository_csv"))
    .withColumn("ingested_at", current_timestamp())
)

products_bronze_df = (
    spark.createDataFrame(products_pandas_df)
    .withColumn("source_system", lit("repository_csv"))
    .withColumn("ingested_at", current_timestamp())
)

display(sales_bronze_df.limit(10))
display(products_bronze_df.limit(10))

In [0]:
(
    sales_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_raw_sales")
)

(
    products_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_raw_products")
)

print("Created workspace.default.bronze_raw_sales")
print("Created workspace.default.bronze_raw_products")

In [0]:
validation_df = spark.sql("""
    SELECT 'bronze_raw_sales' AS table_name, COUNT(*) AS row_count
    FROM workspace.default.bronze_raw_sales

    UNION ALL

    SELECT 'bronze_raw_products' AS table_name, COUNT(*) AS row_count
    FROM workspace.default.bronze_raw_products
""")

display(validation_df)